# EVRS -- Exploratory Data Analysis

Multi-city road risk dataset: **Gandhinagar, Ahmedabad, Surat**.

This notebook is viewer-only -- it reads the already-preprocessed dataset and
just looks at it. No training, no reprocessing. The goal is a quick, honest
feel for what's actually in the data before trusting any model built on it.

**Before running this**, make sure preprocessing has completed at least once:
```bash
python main.py --stage preprocess
```

### Contents
1. Load the data
2. Dataset overview
3. Per-city summary
4. Missing / estimated data
5. Feature distributions
6. What correlates with `risk_score_v1`?
7. Geographic overview -- all 3 cities on one map
8. Per-city road network snapshots (colored by risk)
9. Interactive per-city risk map
10. Notes

## Setup

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import geopandas as gpd
import folium

warnings.filterwarnings("ignore")
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["figure.dpi"] = 100

# repo root is the parent of notebooks/ -- lets us `import config` and reuse
# the same paths/constants as the rest of the pipeline instead of
# hardcoding them again here
sys.path.insert(0, str(Path.cwd().parent))
import config

CITY_NAMES = {"gnr": "Gandhinagar", "ahd": "Ahmedabad", "srt": "Surat"}
CITY_COLORS = {"gnr": "#1E88E5", "ahd": "#E53935", "srt": "#43A047"}

print("Repo root:", config.BASE_DIR)
print("Cities:", CITY_NAMES)

## 1. Load the data

Reads the GeoPackage (not the CSV) because we want `geometry` for the maps later -- the plain CSV drops it since routing/reliability don't need it live from the file.

In [ ]:
if not config.PROCESSED_GPKG.exists():
    raise FileNotFoundError(
        f"{config.PROCESSED_GPKG} not found. Run preprocessing first:\n"
        f"    python main.py --stage preprocess\n"
        f"(or python -m src.data.preprocess) from the repo root, then re-run this cell."
    )

gdf = gpd.read_file(config.PROCESSED_GPKG)
df = gdf.drop(columns="geometry")  # convenience: plain DataFrame view for non-spatial cells

print(f"Loaded {len(gdf):,} road segments across {gdf['city_code'].nunique()} cities")
gdf.head()

## 2. Dataset overview

In [ ]:
print("Shape:", df.shape)
print()
df.info()

In [ ]:
df.describe(include="number").T

## 3. Per-city summary

In [ ]:
city_summary = (
    df.groupby("city_code")
      .agg(
          rows=("osmid", "count"),
          total_length_km=("length", lambda s: s.sum() / 1000),
          avg_length_m=("length", "mean"),
          avg_risk=("risk_score_v1", "mean"),
          avg_speed_kph=("speed_kph", "mean"),
      )
      .round(2)
      .rename(index=CITY_NAMES)
)
city_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

vc = df["city_code"].value_counts()
vc.rename(index=CITY_NAMES).plot(kind="bar", ax=axes[0], color=[CITY_COLORS[c] for c in vc.index])
axes[0].set_title("Road segments per city")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=0)

tot_km = df.groupby("city_code")["length"].sum() / 1000
tot_km.rename(index=CITY_NAMES).plot(kind="bar", ax=axes[1], color=[CITY_COLORS[c] for c in tot_km.index])
axes[1].set_title("Total road length per city (km)")
axes[1].set_ylabel("km")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

## 4. Missing / estimated data

Sparse OSM tagging is expected and already handled by the pipeline (see `*_observed` flag columns) -- this just makes the sparsity visible rather than silently trusting defaults.

In [ ]:
observed_cols = [c for c in ["surface_observed", "lit_observed", "width_observed", "maxspeed_observed"]
                 if c in df.columns]
missing_pct = (100 - df[observed_cols].mean() * 100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
missing_pct.plot(kind="barh", ax=ax, color="#EF5350")
ax.set_xlabel("% of segments missing this tag (defaulted, not directly observed)")
ax.set_title("OSM tag sparsity")
plt.tight_layout()
plt.show()

print(f"lanes_estimated required a fallback (not directly tagged) for "
      f"{df['lanes_was_estimated'].mean() * 100:.1f}% of segments")

## 5. Feature distributions

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for code_, group in df.groupby("city_code"):
    ax.hist(group["risk_score_v1"], bins=40, alpha=0.5, label=CITY_NAMES[code_],
             color=CITY_COLORS[code_], density=True)
ax.set_xlabel("risk_score_v1")
ax.set_ylabel("Density")
ax.set_title("Risk score distribution by city")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
highway_by_city = pd.crosstab(df["highway"], df["city_code"]).rename(columns=CITY_NAMES)
highway_by_city = highway_by_city.loc[highway_by_city.sum(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(9, 5))
highway_by_city.plot(kind="bar", stacked=True, ax=ax,
                      color=[CITY_COLORS[c] for c in df["city_code"].unique()])
ax.set_title("Road type breakdown by city")
ax.set_ylabel("Segment count")
ax.set_xlabel("highway type")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
feature_cols = [c for c in ["speed_kph", "length", "sinuosity", "betweenness_centrality",
                            "avg_node_degree", "lanes_estimated"] if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flat, feature_cols):
    ax.hist(df[col].dropna(), bins=40, color="#5C6BC0", alpha=0.85)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 6. What correlates with `risk_score_v1`?

Sanity-check, not a substitute for the model -- `risk_score_v1` is a hand-built formula (see `src/data/preprocess.py::_compute_risk_score`), so strong correlations here mostly just confirm the formula's own weights rather than revealing anything new. Useful for catching a miscomputed feature, not for drawing conclusions about real-world risk.

In [ ]:
numeric_cols = [c for c in df.select_dtypes(include="number").columns if c != "risk_score_v1"]
corr = df[numeric_cols + ["risk_score_v1"]].corr()["risk_score_v1"].drop("risk_score_v1").dropna().sort_values()

fig, ax = plt.subplots(figsize=(7, 8))
bar_colors = ["#EF5350" if v > 0 else "#42A5F5" for v in corr.values]
corr.plot(kind="barh", ax=ax, color=bar_colors)
ax.set_title("Correlation with risk_score_v1")
ax.set_xlabel("Pearson correlation")
plt.tight_layout()
plt.show()

## 7. Geographic overview -- all 3 cities on one map

Gandhinagar and Ahmedabad are close together; Surat is further south -- zoomed out enough to show all three at once, with a marker + popup per city (segment count, total road length).

In [ ]:
# fast: mean of per-edge centroids, not a full geometric union (which can be
# slow on a full city network and isn't needed just to place a city marker)
city_centers = {}
for code_, group in gdf.groupby("city_code"):
    centroids = group.geometry.centroid
    city_centers[code_] = (centroids.y.mean(), centroids.x.mean())

overview_center = (
    np.mean([v[0] for v in city_centers.values()]),
    np.mean([v[1] for v in city_centers.values()]),
)
m = folium.Map(location=list(overview_center), zoom_start=7, tiles="cartodbpositron")

for code_, (lat, lon) in city_centers.items():
    n_segments = int((gdf["city_code"] == code_).sum())
    total_km = gdf.loc[gdf["city_code"] == code_, "length"].sum() / 1000
    folium.CircleMarker(
        location=[lat, lon], radius=14, color=CITY_COLORS[code_], fill=True,
        fill_color=CITY_COLORS[code_], fill_opacity=0.55,
        popup=folium.Popup(f"<b>{CITY_NAMES[code_]}</b><br>{n_segments:,} segments<br>"
                            f"{total_km:,.0f} km of road", max_width=200),
        tooltip=CITY_NAMES[code_],
    ).add_to(m)
    folium.map.Marker(
        [lat, lon],
        icon=folium.DivIcon(html=f"<div style='font-size:12px;font-weight:700;color:{CITY_COLORS[code_]};"
                                  f"text-shadow:1px 1px 2px white;'>{CITY_NAMES[code_]}</div>"),
    ).add_to(m)

m

## 8. Per-city road network snapshots (colored by risk)

Static (matplotlib), not interactive -- fast even for a full city network with tens of thousands of segments, unlike drawing every segment individually in folium (see the interactive section below, which samples for exactly this reason).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, code_ in zip(axes, gdf["city_code"].unique()):
    city_gdf = gdf[gdf["city_code"] == code_]
    city_gdf.plot(column="risk_score_v1", cmap="RdYlGn_r", linewidth=0.6, ax=ax, legend=True)
    ax.set_title(f"{CITY_NAMES[code_]} -- colored by risk_score_v1")
    ax.set_axis_off()
plt.tight_layout()
plt.show()

## 9. Interactive per-city risk map

Change `CITY_TO_MAP` below and re-run this cell to switch cities. Capped at `MAX_INTERACTIVE_SEGMENTS` -- real city networks can have tens of thousands of edges, and drawing that many individual `PolyLine`s in folium gets slow/unresponsive well before that; the static plots above already show the full network, this is for interactively poking at hotspots.

In [ ]:
CITY_TO_MAP = "gnr"  # "gnr" | "ahd" | "srt"
MAX_INTERACTIVE_SEGMENTS = 8000

city_gdf = gdf[gdf["city_code"] == CITY_TO_MAP]
center = city_centers[CITY_TO_MAP]

if len(city_gdf) > MAX_INTERACTIVE_SEGMENTS:
    print(f"{len(city_gdf):,} segments -- sampling {MAX_INTERACTIVE_SEGMENTS:,} for interactive "
          f"display performance.")
    city_gdf_display = city_gdf.sample(MAX_INTERACTIVE_SEGMENTS, random_state=config.SEED)
else:
    city_gdf_display = city_gdf

m_city = folium.Map(location=list(center), zoom_start=13, tiles="cartodbpositron")

# 5th/95th percentile clip so a handful of extreme outliers don't wash out
# the color scale for everything else
vmin, vmax = city_gdf_display["risk_score_v1"].quantile([0.05, 0.95])

def risk_color(v):
    t = np.clip((v - vmin) / (vmax - vmin + 1e-9), 0, 1)
    r = int(255 * t)
    g = int(255 * (1 - t))
    return f"#{r:02x}{g:02x}00"

for _, row in city_gdf_display.iterrows():
    if row.geometry is None:
        continue
    coords = [(y, x) for x, y in row.geometry.coords]
    folium.PolyLine(
        coords, color=risk_color(row["risk_score_v1"]), weight=3, opacity=0.75,
        tooltip=f"risk={row['risk_score_v1']:.2f} | {row['highway']}",
    ).add_to(m_city)

m_city

## 10. Notes

A few things worth sanity-checking here before trusting any model built on this data (fill in your own observations after running the cells above):

- **Coverage balance** -- do all 3 cities have a comparable amount of data (Section 3), or is one city going to dominate training just by row count?
- **Tag sparsity** -- how much of `lanes`/`surface`/`lit`/`maxspeed` is actually observed vs. defaulted (Section 4)? Heavy reliance on defaults means those features carry less real signal than they look like they do.
- **`risk_score_v1` distribution** -- roughly log-normal / unimodal, or bimodal / lopsided in a way that might need addressing (Section 5)?
- **Road type balance** -- is any one `highway` type so dominant it could bias the model toward that type's typical risk profile (Section 5)?
- **Geography** -- does anything in the per-city risk maps (Section 8) look like a preprocessing artifact rather than a real pattern (e.g. a suspiciously uniform color block, or a hard edge at a city boundary)?